# 02 · Error analysis

The final model's most confident mistakes on the test set, in both directions:
the 20 on-time orders it rated riskiest (false alarms) and the 20 late orders it
rated safest (misses). What do they have in common?

Run after `make evaluate`, which writes `artifacts/test_predictions.csv`. The
model is frozen; nothing here feeds back into it. Columns recorded after t0
(carrier handover, delivery) appear below only to explain what went wrong; they
are never features.

In [1]:
import duckdb
import pandas as pd

from opspilot.config import ARTIFACTS_DIR, DUCKDB_PATH
from opspilot.features.load import load_features
from opspilot.models.evaluate import FINAL

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

N = 20
scores = pd.read_csv(ARTIFACTS_DIR / "test_predictions.csv")
con = duckdb.connect(str(DUCKDB_PATH), read_only=True)
after_t0 = con.sql("""
    select
        orders.order_id,
        date_diff('second', orders.t0, stg.carrier_handover_at) / 86400.0 as handover_days,
        date_diff('second', stg.carrier_handover_at, orders.delivered_at) / 86400.0 as carrier_days,
        date_diff('day', orders.estimated_delivery_date, cast(orders.delivered_at as date)) as days_late
    from fct_orders as orders
    inner join stg_orders as stg using (order_id)
""").df()
con.close()

test = (
    scores[["order_id", "y", FINAL]]
    .rename(columns={FINAL: "risk"})
    .merge(load_features(), on="order_id")
    .merge(after_t0, on="order_id")
)
false_alarms = test[test.y == 0].nlargest(N, "risk")
misses = test[test.y == 1].nsmallest(N, "risk")
print(f"{len(test):,} test orders, late rate {test.y.mean():.2%}")

/Users/incilaldikmen/OpsPilot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


21,949 test orders, late rate 4.05%


## Who they are

In [2]:
PROFILE = [
    "risk",
    "promised_days",
    "promise_slack_days",
    "route_mean_delivery_days",
    "route_late_rate_smoothed",
    "seller_mean_handover_days",
    "seller_late_rate_smoothed",
    "distance_km",
    "total_freight",
    "same_state",
]


def profile(frame):
    return frame[PROFILE].median()


pd.DataFrame(
    {
        "all test orders": profile(test),
        "late orders": profile(test[test.y == 1]),
        f"{N} false alarms": profile(false_alarms),
        f"{N} misses": profile(misses),
    }
).rename_axis("median")

,all test orders,late orders,20 false alarms,20 misses
median,,,,
risk,0.15,0.22,0.56,0.08
promised_days,23.17,20.64,20.02,23.48
promise_slack_days,10.41,5.77,3.11,12.61
route_mean_delivery_days,12.55,15.40,16.86,11.09
route_late_rate_smoothed,0.04,0.08,0.16,0.04
seller_mean_handover_days,2.55,2.69,3.94,2.31
seller_late_rate_smoothed,0.06,0.06,0.10,0.04
distance_km,405.65,621.30,504.47,467.36
total_freight,18.32,19.32,23.45,16.82


In [3]:
def top_values(frame, column, k=4):
    shares = frame[column].fillna("missing").value_counts(normalize=True).head(k)
    return ", ".join(f"{value} {share:.0%}" for value, share in shares.items())


pd.DataFrame(
    {
        group: {
            column: top_values(frame, column)
            for column in ["customer_state", "seller_state", "main_category", "payment_type"]
        }
        for group, frame in [
            ("all test orders", test),
            (f"{N} false alarms", false_alarms),
            (f"{N} misses", misses),
        ]
    }
)

,all test orders,20 false alarms,20 misses
customer_state,"SP 45%, RJ 12%, MG 11%, RS 5%",RJ 100%,"SP 60%, MG 10%, RJ 5%, SC 5%"
seller_state,"SP 72%, PR 7%, MG 7%, RJ 5%",SP 100%,"PR 35%, MG 25%, SP 15%, DF 5%"
main_category,"health_beauty 11%, bed_bath_table 9%, watches_...","furniture_decor 35%, bed_bath_table 25%, baby ...","computers_accessories 20%, housewares 15%, spo..."
payment_type,"credit_card 77%, boleto 18%, voucher 3%, debit...","credit_card 65%, boleto 35%",credit_card 100%


## What happened after approval

In [4]:
pd.DataFrame(
    {
        "all late orders": test[test.y == 1][
            ["handover_days", "carrier_days", "days_late"]
        ].median(),
        f"{N} misses": misses[["handover_days", "carrier_days", "days_late"]].median(),
        "all on-time orders": test[test.y == 0][
            ["handover_days", "carrier_days", "days_late"]
        ].median(),
        f"{N} false alarms": false_alarms[["handover_days", "carrier_days", "days_late"]].median(),
    }
).rename_axis("median, in days")

,all late orders,20 misses,all on-time orders,20 false alarms
"median, in days",,,,
handover_days,2.35,2.89,1.26,0.99
carrier_days,23.79,17.58,6.13,8.11
days_late,6.00,7.50,-14.00,-10.00


In [5]:
COLUMNS = [
    "risk",
    "customer_state",
    "seller_state",
    "promised_days",
    "promise_slack_days",
    "route_mean_delivery_days",
    "seller_mean_handover_days",
    "handover_days",
    "carrier_days",
    "days_late",
]
misses[COLUMNS]

,risk,customer_state,seller_state,promised_days,promise_slack_days,route_mean_delivery_days,seller_mean_handover_days,handover_days,carrier_days,days_late
15033,0.04,SP,PR,37.11,25.69,11.42,1.91,2.75,43.10,8
19884,0.05,SP,DF,19.98,9.27,10.71,1.81,2.60,43.06,25
15473,0.06,RJ,RJ,30.44,23.82,6.62,2.27,2.91,36.32,8
6292,0.06,MG,MG,12.23,3.10,9.13,1.62,1.84,16.26,5
6050,0.07,SP,PR,30.39,18.77,11.62,5.17,34.93,6.07,10
6049,0.07,SP,PR,30.39,18.77,11.62,5.17,39.93,8.00,17
1337,0.07,SP,MG,18.98,7.66,11.32,2.51,23.49,8.09,12
21852,0.07,SC,SC,8.02,-0.58,8.60,1.35,0.67,30.09,22
11863,0.07,PR,PR,15.48,7.15,8.33,2.34,1.13,15.90,1
10633,0.08,SP,PR,23.39,11.96,11.43,2.96,8.91,17.25,2


In [6]:
false_alarms[COLUMNS]

,risk,customer_state,seller_state,promised_days,promise_slack_days,route_mean_delivery_days,seller_mean_handover_days,handover_days,carrier_days,days_late
21143,0.61,RJ,SP,16.29,-0.31,16.59,3.91,1.04,12.16,-4
12119,0.61,RJ,SP,15.86,-1.09,16.95,3.52,0.49,16.20,0
10441,0.60,RJ,SP,16.85,-0.21,17.06,4.03,2.46,12.15,-3
19926,0.59,RJ,SP,19.33,2.69,16.64,3.94,4.97,3.22,-12
3444,0.57,RJ,SP,22.76,6.34,16.42,3.92,1.75,16.57,-5
7703,0.57,RJ,SP,20.28,3.32,16.96,4.13,4.91,6.41,-9
10505,0.57,RJ,SP,16.80,-0.26,17.06,8.86,0.39,7.18,-10
5204,0.57,RJ,SP,22.19,5.40,16.79,4.40,0.01,3.05,-20
4437,0.57,RJ,SP,22.28,5.49,16.78,3.38,0.95,8.21,-14
10581,0.56,RJ,SP,19.52,2.46,17.06,4.02,1.87,12.63,-6


## Conclusion

**The false alarms are one route that recovered.** All 20 are orders from São
Paulo sellers to customers in Rio de Janeiro. The model saw a slow route with a
high late rate (median 16.9 days, 16% late as of t0) and a tight promise (3.1
days of slack). That history was shaped by the February and March 2018 spike,
which hit Rio hardest. From April the route recovered: these orders reached the
carrier in a day, spent about 8 days in transit, and arrived a median 10 days
early. This is the prevalence shift seen through a single route, and the reason
the model's probabilities run high on test.

**The misses looked safe at approval and went wrong afterwards.** They had
generous promises (median 12.6 days of slack), fast routes and quick sellers,
and the model rated them 4% to 9%. What went wrong happened after t0, which the
model cannot see:

- in 6 of the 20 the seller took 9 to 40 days to hand the order to the carrier,
  against its usual 2 to 5;
- in 7 the carrier leg took 30 to 135 days.

Sellers in Paraná are over-represented among the misses (7 of 20, against 7% of
test orders).

**What would help.** Seller delays are visible at carrier handover, days before
the promised date. Re-scoring each order at handover, with the actual handover
delay as a feature, is the clearest next step (v2). Carrier delays of a month or
more look like individual incidents and are unlikely to be predictable from
this data. The recovered-route false alarms argue for retraining on recent
months, which the monthly backtest in `reports/evaluation.md` supports.